# Big Data Analytics - Exercises 1

## Problem 3. Using DataFrames and Spark SQL 12 Points


Team Members:
* Esteban Gerardo Leiva Montenegro (esteban.leiva.001@student.uni.lu)
* Isaac Fabián Palma Medina (isaac.palma.001@student.uni.lu)
* José Valdivia Agüero (jose.valdivia.001@student.uni.lu)



In [28]:
import pyspark
import pandas as pd

print("PySpark version:", pyspark.__version__)

# Initialize the Spark session
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col, coalesce, lit, avg
spark = SparkSession.builder.appName("DataFrame-Example").getOrCreate()

PySpark version: 4.0.2


## Load titanic.csv dataset

Note: Make sure you the dataset is in ./titanic.csv

In [ ]:
# Load and read the CSV file and add options
df = spark.read.option("inferSchema", True) \
                       .option("header", True) \
                       .csv("./titanic.csv")
df

DataFrame[PassengerId: int, Survived: int, Pclass: int, Name: string, Sex: string, Age: double, SibSp: int, Parch: int, Ticket: string, Fare: double, Cabin: string, Embarked: string]

In [30]:
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [31]:
df.toPandas()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,None,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"""Johnston, Miss. Catherine Helen """"Carrie""""""",female,NaN,1,2,W./C. 6607,23.4500,None,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### a)
Create a user-defined function (UDF) called MedianAge that computes the median age over all
passengers in the initial Spark DataFrame. Your function should automatically replace all null
entries for the passengers’ ages with the default integer value “23” while calculating the median age.
Refer to the “User-Defined Aggregate Functions (UDAFs)” section in the Spark SQL programming
guidelines2
for an example.

In [35]:
@pandas_udf("double")
def MedianAge(ages: pd.Series) -> float:
    # use 23 in case of null rows
    return float(ages.fillna(23).median())

median_age_row = df.select(MedianAge(col("Age")).alias("Median_Age")).collect()
median_age_value = median_age_row[0]["Median_Age"]

print(f"The calculated mediange age value is: {median_age_value}")

The calculated mediange age value is: 24.0


# b)
Implement the following query using only built-in DataFrame and/or SQL operations. Refer to the
Spark SQL documentation3
for the list of operations that can be performed on a DataFrame.
Query: “Select the name and fare price of all passengers whose age is greater than the median age
of all passengers who embarked on the Titanic.”  

In [27]:
result_b = df.filter(coalesce(col("Age"), lit(23)) > median_age_value) \
             .select("Name", "Fare")

result_b.show(truncate=False)

+---------------------------------------------------------+-------+
|Name                                                     |Fare   |
+---------------------------------------------------------+-------+
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)      |71.2833|
|Heikkinen, Miss. Laina                                   |7.925  |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)             |53.1   |
|Allen, Mr. William Henry                                 |8.05   |
|McCarthy, Mr. Timothy J                                  |51.8625|
|Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)        |11.1333|
|Bonnell, Miss. Elizabeth                                 |26.55  |
|Andersson, Mr. Anders Johan                              |31.275 |
|Hewlett, Mrs. (Mary D Kingcome)                          |16.0   |
|Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)  |18.0   |
|Fynney, Mr. Joseph J                                     |26.0   |
|Beesley, Mr. Lawrence                          

# (c)

Assume we wish to investigate the following (statistical) hypothesis:

- Hypothesis: “The average fare price of passengers who survived the sinking of the Titanic is not
very different from (i.e., neither much below nor much above) the average fare price of passengers
who did not survive the sinking of the Titanic.”

In [34]:
fare_comparison = df.groupBy("Survived") \
                    .agg(avg("Fare").alias("Average_Fare"))

fare_comparison.show()

+--------+------------------+
|Survived|      Average_Fare|
+--------+------------------+
|       1| 48.39540760233917|
|       0|22.117886885245877|
+--------+------------------+



The hypothesis is *rejected*.

The numerical results show that the average fare for passengers who survived (Survived = 1) is approximately 48.40, whereas the average fare for those who did not survive (Survived = 0) is approximately 22.12.

There is a significant difference between these two groups. Survivors paid, on average, more than double the fare of non-survivors. Therefore, the statement that the average fares are "not very different" is false.